# fy-m1 05 — try fy-m1 (chat)
Chat with a trained fy-m1 adapter. **Runtime → Run all**, then type in the box under the last cell.
An empty line or `quit` ends the chat. Loads the base model in 4-bit plus the LoRA adapter `<RUN_ID>/final/`
from the private `Fan1000-Prog/fyM1-ckpt`, with the training system prompt (private `fyM1-chats`, at the
run's data revision). A 4-bit 9B fits any Colab GPU (T4, L4, A100). Needs the `HF_TOKEN` secret (Colab: Secrets
pane with notebook access) and code synced to `fyM1-ckpt` `code/`. Nothing typed here is saved anywhere.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U unsloth unsloth_zoo

In [ ]:
RUN_ID = "fym1-9b-a100-r1"  # the run whose final/ adapter to load
CKPT_REPO, DATA_REPO = "Fan1000-Prog/fyM1-ckpt", "Fan1000-Prog/fyM1-chats"
TEMPERATURE, TOP_P, MAX_NEW_TOKENS, HISTORY_TURNS = 0.7, 0.9, 256, 6

In [ ]:
# Runtime bootstrap (Colab or Kaggle). The HF token is needed before fym1 can be downloaded, so this cell
# carries a verbatim copy of fym1.runtime.hf_token and its helpers (checked by tests/test_notebook_runtime.py).
import os
import tempfile

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle's T4 runtime may preset "0,1"; always use one GPU
CODE_DIR = tempfile.mkdtemp(prefix="fym1code-")  # fresh each run: nothing stale on sys.path

MISSING_TOKEN = ("no HF_TOKEN found: on Kaggle attach it under Add-ons → Secrets; on Colab add it in the "
                 "Secrets pane with notebook access; elsewhere set the HF_TOKEN environment variable")


def on_kaggle() -> bool:
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
        return True
    try:
        import kaggle_secrets  # noqa: F401  (only installed in Kaggle kernels)
    except ImportError:
        return False
    return True


def _secret(get) -> str | None:
    try:
        return get("HF_TOKEN")
    except Exception:  # missing or not attached: the caller raises one clear error; never echo details
        return None


def hf_token() -> str:
    """HF token from Kaggle secrets, else Colab secrets, else the HF_TOKEN env var. Never print it."""
    if on_kaggle():
        from kaggle_secrets import UserSecretsClient

        token = _secret(UserSecretsClient().get_secret)
    else:
        try:
            from google.colab import userdata
        except ImportError:
            token = os.environ.get("HF_TOKEN")
        else:
            token = _secret(userdata.get)
    if not token:
        raise RuntimeError(MISSING_TOKEN)
    return token

In [ ]:
import sys, json
from huggingface_hub import hf_hub_download, snapshot_download

os.environ["HF_TOKEN"] = hf_token()  # never print it
code_dir = snapshot_download(CKPT_REPO, allow_patterns=["code/**"], local_dir=CODE_DIR)
sys.path.insert(0, f"{code_dir}/code/src")
from fym1.train.chat_format import CHAT_TEMPLATE_KWARGS, training_text
from fym1.train.run_state import check_hub_id
from fym1.runtime import scratch_dir

train_log = json.load(open(hf_hub_download(CKPT_REPO, f"{RUN_ID}/train_log.json")))
MODEL_ID, DATA_REVISION = train_log["model"], train_log["config"]["data_revision"]
system_prompt = open(hf_hub_download(DATA_REPO, "system_prompt.txt", repo_type="dataset",
                                     revision=DATA_REVISION)).read().strip()
adapter_dir = snapshot_download(CKPT_REPO, allow_patterns=[f"{RUN_ID}/final/*"], local_dir=f"{scratch_dir()}/try")
ADAPTER = f"{adapter_dir}/{RUN_ID}/final"
print("RUN", RUN_ID, "| base", MODEL_ID, "| adapter", train_log["final_adapter"])

In [ ]:
import torch
from peft import PeftModel
from unsloth import FastModel

model, processor = FastModel.from_pretrained(MODEL_ID, max_seq_length=2048, load_in_4bit=True,
                                             full_finetuning=False, use_exact_model_name=True)
check_hub_id("loaded model", model.config._name_or_path, MODEL_ID)
model = PeftModel.from_pretrained(model, ADAPTER)
FastModel.for_inference(model)
tok = getattr(processor, "tokenizer", processor)
print("READY on", torch.cuda.get_device_name(0), "| vram GB", round(torch.cuda.memory_allocated() / 2**30, 2))

In [ ]:
def render(messages, add_generation_prompt=False):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=add_generation_prompt,
                                   **CHAT_TEMPLATE_KWARGS)


def chat_prompt(messages):
    """Past turns exactly as training rendered them (training_text), then the generation prompt."""
    return training_text(render, messages) + render(messages, True)[len(render(messages)):]

In [ ]:
from transformers import TextStreamer


@torch.inference_mode()
def reply(history):
    messages = [{"role": "system", "content": system_prompt}] + history[-2 * HISTORY_TURNS + 1:]
    text = chat_prompt(messages)
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    ids = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE,
                         top_p=TOP_P, pad_token_id=tok.pad_token_id,
                         streamer=TextStreamer(tok, skip_prompt=True, skip_special_tokens=True))
    return tok.decode(ids[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()


history = []
while True:
    user = input("ianao> ").strip()
    if not user or user.lower() == "quit":
        print("veloma!")
        break
    history.append({"role": "user", "content": user})
    print("fy-m1> ", end="")
    history.append({"role": "assistant", "content": reply(history)})